# Semantic Search System — Analysis Notebook

This notebook presents the analysis for Parts 2 and 3:
- **Part 2**: Fuzzy clustering — K selection, cluster semantics, boundary analysis
- **Part 3**: Semantic cache — threshold exploration and trade-off analysis

Run `python -m scripts.prepare_data` and `python -m scripts.analyze_threshold` before this notebook.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
DATA_DIR = Path('data')

# Load artifacts
embeddings = np.load(DATA_DIR / 'embeddings.npy')
labels = np.load(DATA_DIR / 'labels.npy')
cluster_distributions = np.load(DATA_DIR / 'cluster_distributions.npy')
cluster_label_mat = np.load(DATA_DIR / 'cluster_label_matrix.npy')

with open(DATA_DIR / 'target_names.json') as f:
    target_names = json.load(f)
with open(DATA_DIR / 'k_sweep_results.json') as f:
    k_sweep = json.load(f)

# Load texts
texts = []
with open(DATA_DIR / 'cleaned_texts.jsonl') as f:
    for line in f:
        texts.append(json.loads(line))

print(f'Loaded {len(texts)} documents, {cluster_distributions.shape[1]} clusters')

## Part 2.2 — Choosing K

We swept K from 10 to 30 and computed BIC (penalised likelihood) and silhouette score.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

k_vals = k_sweep['k_values']
bic = k_sweep['bic_scores']
sil = k_sweep['silhouette_scores']

ax1.plot(k_vals, bic, 'bo-', linewidth=2)
ax1.set_xlabel('K (number of clusters)')
ax1.set_ylabel('BIC (lower is better)')
ax1.set_title('BIC vs K — Gaussian Mixture Model')
best_k_bic = k_vals[np.argmin(bic)]
ax1.axvline(x=best_k_bic, color='r', linestyle='--', label=f'Best K={best_k_bic}')
ax1.legend()

ax2.plot(k_vals, sil, 'go-', linewidth=2)
ax2.set_xlabel('K (number of clusters)')
ax2.set_ylabel('Silhouette Score (higher is better)')
ax2.set_title('Silhouette Score vs K')
best_k_sil = k_vals[np.argmax(sil)]
ax2.axvline(x=best_k_sil, color='r', linestyle='--', label=f'Best K={best_k_sil}')
ax2.legend()

plt.tight_layout()
plt.savefig(DATA_DIR / 'k_selection.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'BIC-optimal K: {best_k_bic}')
print(f'Silhouette-optimal K: {best_k_sil}')

## Part 2.3 — Cluster Analysis

### Cluster-Label Co-occurrence Heatmap

This shows how much probability mass from each ground-truth label lands in each cluster.
Strong diagonal presence would mean clusters align with newsgroup labels.
Off-diagonal mass reveals where the semantic structure diverges from the labelling scheme.

In [ ]:
# Normalise by column (per label) to show distribution across clusters
col_sums = cluster_label_mat.sum(axis=0, keepdims=True)
col_sums[col_sums == 0] = 1
normalised = cluster_label_mat / col_sums

fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(
    normalised,
    xticklabels=[n.split('.')[-1] for n in target_names],
    yticklabels=[f'C{i}' for i in range(normalised.shape[0])],
    cmap='YlOrRd',
    ax=ax,
    annot=False,
)
ax.set_xlabel('Ground Truth Label')
ax.set_ylabel('GMM Cluster')
ax.set_title('Cluster-Label Co-occurrence (column-normalised)')
plt.tight_layout()
plt.savefig(DATA_DIR / 'cluster_label_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### Per-Cluster Analysis: Top Documents and Dominant Labels

In [ ]:
n_clusters = cluster_distributions.shape[1]

for c in range(n_clusters):
    probs = cluster_distributions[:, c]
    top_indices = np.argsort(probs)[-5:][::-1]
    
    # Dominant labels in this cluster
    label_mass = cluster_label_mat[c]
    top_labels_idx = np.argsort(label_mass)[-3:][::-1]
    top_labels = [(target_names[l], round(label_mass[l], 1)) for l in top_labels_idx]
    
    print(f'\n{"="*60}')
    print(f'CLUSTER {c} — Top labels: {top_labels}')
    print(f'{"="*60}')
    for idx in top_indices:
        print(f'  P={probs[idx]:.3f} | {target_names[labels[idx]]:30s} | {texts[idx][:120]}...')

### Boundary Documents — High-Entropy Cases

These are documents whose cluster distribution has high Shannon entropy,
meaning the model is genuinely uncertain about where they belong.
These boundary cases are often the most semantically interesting.

In [ ]:
# Compute entropy
p = np.clip(cluster_distributions, 1e-10, 1.0)
entropy = -np.sum(p * np.log2(p), axis=1)

# Plot entropy distribution
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(entropy, bins=50, color='steelblue', edgecolor='white')
ax.set_xlabel('Shannon Entropy (bits)')
ax.set_ylabel('Number of Documents')
ax.set_title('Distribution of Cluster Assignment Entropy')
ax.axvline(x=np.percentile(entropy, 95), color='red', linestyle='--',
           label=f'95th percentile = {np.percentile(entropy, 95):.2f}')
ax.legend()
plt.tight_layout()
plt.savefig(DATA_DIR / 'entropy_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mean entropy: {entropy.mean():.3f}')
print(f'Median entropy: {np.median(entropy):.3f}')
print(f'95th percentile: {np.percentile(entropy, 95):.3f}')

In [ ]:
# Show top 10 boundary cases
boundary_indices = np.argsort(entropy)[-10:][::-1]

print('TOP 10 BOUNDARY DOCUMENTS (highest entropy):')
print('=' * 80)
for idx in boundary_indices:
    top2_clusters = np.argsort(cluster_distributions[idx])[-2:][::-1]
    top2_probs = [cluster_distributions[idx][c] for c in top2_clusters]
    
    print(f'\nDoc index: {idx} | Entropy: {entropy[idx]:.3f}')
    print(f'  Ground truth: {target_names[labels[idx]]}')
    print(f'  Top cluster: C{top2_clusters[0]} (P={top2_probs[0]:.3f})')
    print(f'  2nd cluster: C{top2_clusters[1]} (P={top2_probs[1]:.3f})')
    print(f'  Text: {texts[idx][:200]}...')
    print(f'  ---')

## Part 3 — Semantic Cache Threshold Analysis

The similarity threshold is the single most important tunable in the cache.
Below we visualise the precision/recall/F1 trade-off across threshold values.

In [ ]:
try:
    with open(DATA_DIR / 'threshold_analysis.json') as f:
        threshold_results = json.load(f)
except FileNotFoundError:
    print('Run `python -m scripts.analyze_threshold` first.')
    threshold_results = None

if threshold_results:
    thresholds = [r['threshold'] for r in threshold_results]
    precisions = [r['precision'] for r in threshold_results]
    recalls = [r['recall'] for r in threshold_results]
    f1s = [r['f1'] for r in threshold_results]
    fp_rates = [r['false_positive_rate'] for r in threshold_results]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.plot(thresholds, precisions, 'b^-', label='Precision', linewidth=2)
    ax1.plot(thresholds, recalls, 'rs-', label='Recall', linewidth=2)
    ax1.plot(thresholds, f1s, 'go-', label='F1', linewidth=2, markersize=8)
    ax1.set_xlabel('Similarity Threshold')
    ax1.set_ylabel('Score')
    ax1.set_title('Cache Performance vs Threshold')
    ax1.legend()
    ax1.set_ylim(0, 1.05)
    
    # Mark best F1
    best_idx = np.argmax(f1s)
    ax1.axvline(x=thresholds[best_idx], color='gray', linestyle=':', alpha=0.5)
    ax1.annotate(f'Best F1={f1s[best_idx]:.3f}\n@ θ={thresholds[best_idx]:.2f}',
                 xy=(thresholds[best_idx], f1s[best_idx]),
                 xytext=(thresholds[best_idx]+0.05, f1s[best_idx]-0.15),
                 arrowprops=dict(arrowstyle='->', color='gray'),
                 fontsize=10)
    
    ax2.plot(thresholds, fp_rates, 'r^-', label='False Positive Rate', linewidth=2)
    ax2.set_xlabel('Similarity Threshold')
    ax2.set_ylabel('Rate')
    ax2.set_title('False Positive Rate vs Threshold')
    ax2.legend()
    ax2.set_ylim(-0.05, 1.05)
    
    plt.tight_layout()
    plt.savefig(DATA_DIR / 'threshold_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print('\nDetailed results:')
    print(f'{"Threshold":>10} {"Precision":>10} {"Recall":>10} {"F1":>10} {"FP Rate":>10}')
    print('-' * 52)
    for r in threshold_results:
        print(f'{r["threshold"]:>10.2f} {r["precision"]:>10.4f} {r["recall"]:>10.4f} '
              f'{r["f1"]:>10.4f} {r["false_positive_rate"]:>10.4f}')

## Interpretation

### What the threshold reveals about the system:

1. **Low threshold (0.70–0.75)**: The cache is aggressive — it matches almost everything.
   Hit rate is high, but so is the false positive rate. Queries about unrelated topics
   get served cached results from vaguely similar queries. This is the "everything looks
   the same to the model" regime.

2. **Medium threshold (0.80–0.90)**: The sweet spot. The cache correctly identifies
   paraphrases while rejecting genuinely different queries. The embedding model's
   semantic discrimination is strongest in this range.

3. **High threshold (0.95+)**: The cache becomes nearly useless — only near-identical
   phrasings trigger a hit. This defeats the purpose of a *semantic* cache.

### Why cluster-based pruning matters:

Without cluster pruning, the cache would compare against ALL stored embeddings.
With K clusters, the average lookup only compares against ~1/K of entries.
This makes the cache scalable to thousands of entries without sacrificing latency.